# 09 - Figuras y tablas para la memoria

Este notebook recopila las salidas más útiles y genera una carpeta final con tablas y figuras que seguramente usaré en LaTeX.

In [2]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEVEL_ORDER = {"A1": 1, "A2": 2, "B1": 3, "B2": 4, "C1": 5, "C2": 6}
LEVEL_ORDER_INV = {v: k for k, v in LEVEL_ORDER.items()}
RADON_GRADE_ORDER = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6}
RADON_GRADE_ORDER_INV = {v: k for k, v in RADON_GRADE_ORDER.items()}

def clean_file_name(path):
    """Devuelve un nombre de fichero comparable entre herramientas."""
    return Path(str(path)).name

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def detect_json_type(path):
    """Intenta detectar si un JSON parece de Radon o de PyCEFR."""
    try:
        data = load_json(path)
    except Exception:
        return "unknown"
    # Buscar un registro de ejemplo dentro del árbol project/file/[records]
    for project, files in data.items():
        if not isinstance(files, dict):
            continue
        for file_path, records in files.items():
            if isinstance(records, list) and records:
                rec = records[0]
                if isinstance(rec, dict):
                    if {"Class", "Start Line", "End Line", "Level"}.issubset(set(rec.keys())):
                        return "pycefr"
                    if {"type", "rank", "complexity", "lineno", "endline"}.issubset(set(rec.keys())):
                        return "radon"
    return "unknown"

In [3]:
MEMORIA_DIR = OUTPUT_DIR / "memoria"
MEMORIA_DIR.mkdir(exist_ok=True)

files_to_copy = [
    "02_pycefr_distribucion_niveles.png",
    "02_pycefr_top_constructos.png",
    "03_radon_distribucion_calificaciones.png",
    "03_radon_hist_complejidad.png",
    "04_radon_vs_pycefr_scatter.png",
    "02_pycefr_resumen_por_caso.csv",
    "03_radon_resumen_por_caso.csv",
    "04_tabla_cruce_radon_pycefr.csv",
    "05_ejemplos_para_memoria.csv",
    "08_proyectos_analizados.csv",
]

import shutil
for name in files_to_copy:
    src = OUTPUT_DIR / name
    if src.exists():
        shutil.copy(src, MEMORIA_DIR / name)
        print("Copiado:", name)
    else:
        print("No existe todavía:", name)

Copiado: 02_pycefr_distribucion_niveles.png
Copiado: 02_pycefr_top_constructos.png
Copiado: 03_radon_distribucion_ranks.png
Copiado: 03_radon_hist_complejidad.png
Copiado: 04_radon_vs_pycefr_scatter.png
Copiado: 02_pycefr_resumen_por_caso.csv
Copiado: 03_radon_resumen_por_caso.csv
Copiado: 04_tabla_concordancia_rank_nivel.csv
Copiado: 05_ejemplos_para_memoria.csv
Copiado: 08_corpus_trazabilidad.csv


In [8]:
# Pequeña ayuda para insertar figuras en LaTeX

latex_examples = []

for fig in MEMORIA_DIR.glob("*.png"):
    label = fig.stem.replace("_", "-")
    caption = fig.stem.replace("_", " ")

    latex_examples.append(fr"""
\begin{{figure}}[h]
    \centering
    \includegraphics[width=0.85\textwidth]{{figures/{fig.name}}}
    \caption{{TODO: describir {caption}}}
    \label{{fig:{label}}}
\end{{figure}}
""".strip())

latex_text = "\n\n".join(latex_examples)

latex_path = Path(MEMORIA_DIR) / "latex_figuras_ejemplo.tex"
latex_path.write_text(latex_text, encoding="utf-8")

print("Generado ejemplo LaTeX:", latex_path)

Generado ejemplo LaTeX: /home/juan/Documents/Analisis-CC-PyCEFR/outputs/memoria/latex_figuras_ejemplo.tex


## Comentario final

La carpeta `outputs/memoria` contiene una selección de tablas y figuras que pueden copiarse a la plantilla LaTeX del TFG. La idea es no perder tiempo buscando resultados repartidos por todo el proyecto.